In [187]:
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-community langchain-huggingface numpy --quiet --no-cache

In [188]:
from dotenv import load_dotenv
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from warnings import filterwarnings
filterwarnings('ignore') #This is the ending over here!!!
load_dotenv()

False

In [189]:
if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")
#This is the ending of the first if case over here!!!
print("Setup complete")

Setup complete


In [190]:
def DRY0(List):
  for rank, (idx, score, doc) in enumerate(List, 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f} : {doc}") #This is the ending of the DRY0 function over here

In [191]:
def ResultDisplay(String1, query, List1):
  print(String1)#This is the first case over here!!
  print(f"Query: {query}") #Now, we would be taking the help of the DRY principle to ease this all out!!
  DRY0(List1)

In [192]:
def get_score(scores, top_k):
  return np.argsort(scores)[::-1][:top_k]

In [193]:
def dense_retrieve(query, corpus):
  #This part talks about the dense_retrieval over here!!!
  model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
  encodings = model.encode(corpus, convert_to_numpy= True)
  corpus_embeddings = encodings / np.linalg.norm(encodings, axis=1, keepdims=True)
  q_vec = model.encode([query], convert_to_numpy= True)[0]
  scores = corpus_embeddings @ q_vec / (np.linalg.norm(corpus_embeddings, axis=1) * np.linalg.norm(q_vec))
  return [(i, scores[i], corpus[i]) for i in get_score(scores, 3)]#This is teh first ending over here

In [194]:
def sparse_retrieve(query, corpus):
  #Here, we are using the help of the BM25 algorithm over here!!!
  model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
  tokenized_corpus = [doc.lower().split() for doc in corpus] #This is the list over here
  tokenized_query = query.lower().split() #This is teh next part over here
  bm25 = BM25Okapi(tokenized_corpus)
  scores = bm25.get_scores(tokenized_query)
  return [(i, scores[i], corpus[i]) for i in get_score(scores, 3)]

In [195]:
def DRY1(String, ranked, query, corpus):
  ResultDisplay(String, query, ranked) #This is being sent over here

In [196]:
def helper_function(query, corpus, String):
  ranked = dense_retrieve(query, corpus)
  DRY1(String, ranked, query, corpus)

In [197]:
def next_helper_function(query, corpus, String):
  ranked = sparse_retrieve(query, corpus)
  DRY1(String, ranked, query, corpus)

In [198]:
def get_tokens(a):
  return a.lower().split() #This is the ending part over here!!
def get_embeddings(a, encoder):
  return encoder.encode(a, convert_to_numpy=True) #This is the DRY principle in play over here!!!
def normalize1(a):
  return a / np.linalg.norm(a, axis=1, keepdims=True) #This is the next ending over here!!!
def colbert_score(query: str, document: str)-> float:
  query_tokens = get_tokens(query) #This is the first part over here!!!
  doc_tokens = get_tokens(document)
  encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
  query_embeddings = get_embeddings(query_tokens, encoder)
  doc_embeddings = get_embeddings(doc_tokens, encoder)
  sim_matrix = query_embeddings @ doc_embeddings.T
  max_sims = sim_matrix.max(axis=1) #This would be returned over here
  return float(max_sims.sum()) #Return value

In [199]:
def SBERT(corpus, query):
  tokenized_corpus = [doc.lower().split() for doc in corpus]
  bm25 = BM25Okapi(tokenized_corpus)
  bm25_scores = bm25.get_scores(query.lower().split())
  print("STEP 1: BM25 Raw Scores")
  print("-" * 60)
  for i, s in enumerate(bm25_scores):
      print(f"  doc_{i}: {s:.4f}  |  {corpus[i][:60]}")
  bm25_ranked = np.argsort(bm25_scores)[::-1]
  bm25_ranks  = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked)}
  print("\nBM25 Ranking (best → worst):", list(bm25_ranked))
  sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
  doc_vecs   = sbert.encode(corpus, convert_to_numpy=True)
  query_vec  = sbert.encode([query], convert_to_numpy=True)[0]
  doc_vecs_norm   = doc_vecs   / np.linalg.norm(doc_vecs,   axis=1, keepdims=True)
  query_vec_norm  = query_vec  / np.linalg.norm(query_vec)
  sbert_scores = doc_vecs_norm @ query_vec_norm
  print("STEP 2: SBERT Cosine Scores")
  print("-" * 60)
  for i, s in enumerate(sbert_scores):
    print(f"  doc_{i}: {s:.4f}  |  {corpus[i][:60]}")
  sbert_ranked = np.argsort(sbert_scores)[::-1]
  sbert_ranks  = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked)}
  print("\nSBERT Ranking (best → worst):", list(sbert_ranked))
  K = 60  # RRF smoothing constant
  print("STEP 3: Reciprocal Rank Fusion (k=60)")
  print("-" * 80)
  print(f"{'Doc':<8} {'BM25 rank':<12} {'SBERT rank':<13} {'RRF(BM25)':<14} {'RRF(SBERT)':<14} {'Total RRF'}")
  print("-" * 80)
  rrf_scores = {}
  for doc_id in range(len(corpus)):
    rb = bm25_ranks[doc_id]
    rs = sbert_ranks[doc_id]
    rrf_bm25  = 1.0 / (K + rb)
    rrf_sbert = 1.0 / (K + rs)
    total     = rrf_bm25 + rrf_sbert
    rrf_scores[doc_id] = total
    print(f"  doc_{doc_id}  {rb:<12} {rs:<13} {rrf_bm25:<14.6f} {rrf_sbert:<14.6f} {total:.6f}")
  # Final ranking
  final_ranked = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
  print("\n" + "=" * 80)
  print("FINAL HYBRID RANKING")
  print("=" * 80)
  for rank, doc_id in enumerate(final_ranked, 1):
    print(f"  #{rank}  doc_{doc_id} (RRF={rrf_scores[doc_id]:.6f}): {corpus[doc_id]}")

In [200]:
corpus = ["Transformers use self-attention mechanisms to process sequences in parallel", "BERT is a bidirectional encoder trained using masked language modelling", "The BM25 algorithm ranks documents based on the term frequency and the inverse document frequency", "Gradient descent is used in the back-propogation", "Neural networks adjust weights during the backpropogation"]
query_vocab_mismatch = "How do the neural networks learn?" #This is just for the dummy part over here!!!
query_exact_match = "BM25 term frequency" #This is for the sparse retrieval part over here!!!
helper_function(query_vocab_mismatch, corpus, "== This is the dense retrieval over here ==")
helper_function(query_exact_match, corpus, "")
next_helper_function(query_vocab_mismatch, corpus, "== This is the sparse retrieval over here ==")
next_helper_function(query_exact_match, corpus, "") #This is the endiong of the first parts over here
print("== ColBERT ==")
print(f"Query: {query_vocab_mismatch}")
scores = list()
for i, doc in enumerate(corpus):
  s = colbert_score(query_vocab_mismatch, doc)
  scores.append(s)
  print(f" doc_{i} score={s}: {doc}")

best = np.argmax(scores)
print(f" Top_document_{best} - {corpus[best]}") #This is teh ending over here!!!
SBERT(corpus, query_vocab_mismatch)
SBERT(corpus, query_exact_match)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


== This is the dense retrieval over here ==
Query: How do the neural networks learn?
  Rank 1 [doc_4] score=0.5107 : Neural networks adjust weights during the backpropogation
  Rank 2 [doc_3] score=0.4378 : Gradient descent is used in the back-propogation
  Rank 3 [doc_1] score=0.2656 : BERT is a bidirectional encoder trained using masked language modelling


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


STEP 2: SBERT Cosine Scores
------------------------------------------------------------
  doc_0: 0.2623  |  Transformers use self-attention mechanisms to process sequen
  doc_1: 0.2656  |  BERT is a bidirectional encoder trained using masked languag
  doc_2: 0.0990  |  The BM25 algorithm ranks documents based on the term frequen
  doc_3: 0.4378  |  Gradient descent is used in the back-propogation
  doc_4: 0.5107  |  Neural networks adjust weights during the backpropogation

SBERT Ranking (best → worst): [np.int64(4), np.int64(3), np.int64(1), np.int64(0), np.int64(2)]
STEP 3: Reciprocal Rank Fusion (k=60)
--------------------------------------------------------------------------------
Doc      BM25 rank    SBERT rank    RRF(BM25)      RRF(SBERT)     Total RRF
--------------------------------------------------------------------------------
  doc_0  5            4             0.015385       0.015625       0.031010
  doc_1  4            3             0.015625       0.015873       0.03149

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


STEP 2: SBERT Cosine Scores
------------------------------------------------------------
  doc_0: 0.1517  |  Transformers use self-attention mechanisms to process sequen
  doc_1: 0.1039  |  BERT is a bidirectional encoder trained using masked languag
  doc_2: 0.5239  |  The BM25 algorithm ranks documents based on the term frequen
  doc_3: 0.1868  |  Gradient descent is used in the back-propogation
  doc_4: 0.0966  |  Neural networks adjust weights during the backpropogation

SBERT Ranking (best → worst): [np.int64(2), np.int64(3), np.int64(0), np.int64(1), np.int64(4)]
STEP 3: Reciprocal Rank Fusion (k=60)
--------------------------------------------------------------------------------
Doc      BM25 rank    SBERT rank    RRF(BM25)      RRF(SBERT)     Total RRF
--------------------------------------------------------------------------------
  doc_0  5            3             0.015385       0.015873       0.031258
  doc_1  4            4             0.015625       0.015625       0.03125

In [201]:
class HybridRetriever:
    """
    A production-style Hybrid Retriever combining BM25 (sparse) + SBERT (dense)
    using Reciprocal Rank Fusion.
    """
    def __init__(self, corpus: list[str], sbert_model: str = "sentence-transformers/all-MiniLM-L6-v2", k: int = 60):
        self.corpus = corpus
        self.k = k
        # --- Build BM25 index ---
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25  = BM25Okapi(tokenized)
        # --- Build SBERT index ---
        self.sbert     = SentenceTransformer(sbert_model)
        doc_vecs       = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs  = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)
    def retrieve(self, query: str, top_k: int = 3) -> list[dict]:
        # BM25
        bm25_scores  = self.bm25.get_scores(query.lower().split())
        bm25_ranked  = np.argsort(bm25_scores)[::-1]
        bm25_ranks   = {int(doc_id): rank+1 for rank, doc_id in enumerate(bm25_ranked)}
        # SBERT
        q_vec        = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec        = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(doc_id): rank+1 for rank, doc_id in enumerate(sbert_ranked)}
        # RRF Fusion
        rrf = {}
        for doc_id in range(len(self.corpus)):
            rrf[doc_id] = 1/(self.k + bm25_ranks[doc_id]) + 1/(self.k + sbert_ranks[doc_id])
        final = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [{"doc_id": doc_id, "rrf_score": rrf[doc_id], "text": self.corpus[doc_id]} for doc_id in final]
# Demo
retriever = HybridRetriever(corpus)
for q in ["How do neural nets learn?", "BM25 term frequency ranking", "attention in transformers"]:
    print(f"Query: '{q}'")
    results = retriever.retrieve(q, top_k=2)
    for r in results:
        print(f"  → doc_{r['doc_id']} (RRF={r['rrf_score']:.5f}): {r['text']}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: 'How do neural nets learn?'
  → doc_4 (RRF=0.03279): Neural networks adjust weights during the backpropogation
  → doc_3 (RRF=0.03226): Gradient descent is used in the back-propogation

Query: 'BM25 term frequency ranking'
  → doc_2 (RRF=0.03279): The BM25 algorithm ranks documents based on the term frequency and the inverse document frequency
  → doc_3 (RRF=0.03200): Gradient descent is used in the back-propogation

Query: 'attention in transformers'
  → doc_0 (RRF=0.03279): Transformers use self-attention mechanisms to process sequences in parallel
  → doc_3 (RRF=0.03200): Gradient descent is used in the back-propogation

